# Module 3 — Asset Allocation Engine

**Theory: Markowitz Mean-Variance Optimization (1952)**

Given N assets with expected returns μ and covariance Σ, find weights w that:
- **Maximise** expected return for a given level of risk, OR
- **Minimise** variance for a given return target

The set of all such optimal portfolios traces the **Efficient Frontier**.

**Asset universe (13 instruments):**
Large/Mid/Small cap equity · International equity · Gold · Silver ·
Govt bonds · Corporate bonds · ELSS · FD · PPF · RBI Bond · Liquid fund

In [ ]:
# ── Setup (Colab) ────────────────────────────────────────────────
!pip install -q numpy pandas PyPortfolioOpt plotly yfinance
import os
if not os.path.exists('robo-advisor'):
    !git clone https://github.com/parinmody30/robo-advisor.git
else:
    !git -C robo-advisor pull

import sys
sys.path.insert(0, 'robo-advisor/src')
DATA_DIR = 'robo-advisor/data'
print('Setup complete.')

In [ ]:
import json, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

from data_loader import load_all
from optimizer import run_optimizer, generate_frontier, total_equity_weight, ASSET_LABELS, ASSET_COLORS

# Load risk profile
try:
    with open(f'{DATA_DIR}/risk_profile.json') as f:
        profile = json.load(f)
    persona       = profile['persona']
    target_return = profile['target_return']
    print(f'Profile: {persona} | Target return: {target_return*100:.0f}% p.a.')
except FileNotFoundError:
    persona, target_return = 'Balanced', 0.12
    print('No saved profile — using Balanced / 12%')

## 3A. Fetch market data and compute returns
*(Pulls ~10 years of data — takes ~1 minute)*

In [ ]:
prices, mu, cov = load_all(start='2015-01-01')

print('Assets loaded:', list(mu.index))
print()
print('Annualised Expected Returns:')
for asset, ret in mu.items():
    label = ASSET_LABELS.get(asset, asset)
    print(f'  {label:<30} {ret*100:>6.2f}%')

## 3B. Correlation heatmap
High correlation between assets reduces diversification benefit.

In [ ]:
import plotly.graph_objects as go

# Only show live assets (exclude synthetics)
live_assets = [a for a in cov.index if a not in ['FixedDeposit','PPF','RBI_Bond','LiquidFund']]
corr = cov.loc[live_assets, live_assets].copy()
for a in live_assets:
    corr[a] = corr[a] / (np.sqrt(cov.loc[a,a]) * np.sqrt(cov.loc[live_assets, live_assets].values.diagonal()))

labels = [ASSET_LABELS.get(a, a) for a in live_assets]
fig = go.Figure(go.Heatmap(
    z=corr.values, x=labels, y=labels,
    colorscale='RdYlGn', zmid=0, zmin=-1, zmax=1,
    text=np.round(corr.values, 2), texttemplate='%{text}',
))
fig.update_layout(title='Asset Correlation Matrix',
                  height=500, template='plotly_white')
fig.show()

## 3C. Run the optimizer

In [ ]:
result = run_optimizer(mu, cov, persona, target_return)

print(f'Optimal Portfolio — {persona}')
print('=' * 45)
for asset, weight in sorted(result['weights'].items(), key=lambda x: -x[1]):
    label = ASSET_LABELS.get(asset, asset)
    bar   = '█' * int(weight * 50)
    print(f'  {label:<28} {weight*100:>5.1f}%  {bar}')
print('=' * 45)
print(f'  Expected Return  : {result["expected_return"]*100:.1f}%')
print(f'  Volatility       : {result["volatility"]*100:.1f}%')
print(f'  Sharpe Ratio     : {result["sharpe_ratio"]:.2f}')
eq_pct = total_equity_weight(result['weights']) * 100
print(f'  Total Equity     : {eq_pct:.1f}%')

## 3D. Allocation pie chart

In [ ]:
weights = result['weights']
labels  = [ASSET_LABELS.get(k, k) for k in weights]
values  = list(weights.values())
colors  = [ASSET_COLORS.get(k, '#888') for k in weights]

fig = go.Figure(go.Pie(
    labels=labels, values=values, marker_colors=colors,
    textinfo='label+percent', hole=0.35,
))
fig.update_layout(
    title=f'Optimal Allocation — {persona} (Return: {result["expected_return"]*100:.1f}%, Sharpe: {result["sharpe_ratio"]:.2f})',
    template='plotly_white', height=520,
)
fig.show()

## 3E. Efficient frontier
Your portfolio is marked on the curve. Every point above-left is unachievable; every point below-right is suboptimal.

In [ ]:
print('Generating efficient frontier (may take ~30s)...')
frontier = generate_frontier(mu, cov, persona)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=frontier['volatility']*100, y=frontier['return']*100,
    mode='lines', name='Efficient Frontier',
    line=dict(color='#1565C0', width=3),
))
fig.add_trace(go.Scatter(
    x=[result['volatility']*100], y=[result['expected_return']*100],
    mode='markers+text', name='Your Portfolio',
    marker=dict(size=14, color='#E53935', symbol='star'),
    text=[f'{persona}'], textposition='top right',
))
fig.update_layout(
    title='Efficient Frontier — Risk vs Return',
    xaxis_title='Volatility (Risk) %',
    yaxis_title='Expected Return %',
    template='plotly_white', height=480,
)
fig.show()

## 3F. Save allocation for downstream modules

In [ ]:
import os
os.makedirs(DATA_DIR, exist_ok=True)

output = {
    'persona':          persona,
    'target_return':    target_return,
    'weights':          result['weights'],
    'expected_return':  result['expected_return'],
    'volatility':       result['volatility'],
    'sharpe_ratio':     result['sharpe_ratio'],
    'total_equity_pct': round(total_equity_weight(result['weights']) * 100, 2),
}
with open(f'{DATA_DIR}/allocation.json', 'w') as f:
    json.dump(output, f, indent=2)
print(f'Allocation saved to {DATA_DIR}/allocation.json')

## Theory note — where MVO breaks (and how we fix it)

| Assumption | Reality | Our fix |
|---|---|---|
| Returns are normally distributed | Fat tails — crashes are more frequent and severe than normal distribution predicts | Module 4: Monte Carlo with historical bootstrapping |
| Correlations are stable | Correlations spike toward 1 in crises — diversification fails exactly when you need it | Stress scenario injection in Module 4 |
| Expected returns are known | They're estimated from history — garbage in, garbage out | Optional: Black-Litterman blends market priors with views |
| No constraints | Unconstrained MVO gives absurd weights (100% in one asset) | We apply per-asset min/max bounds per persona |